# Cleaning danych

Notebook roboczy do przygotowania danych.


## 1. Importy, konfiguracja i funkcje pomocnicze


In [61]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_FILE = Path("tauron_liga_statystyki.csv")
OUTPUT_FILE = Path("processed_data.csv")

# MATCH_TYPE_FIXES = {
#     "3.sty": "3:1",
#     "3.lut": "3:2",
#     "3-0": "3:0",
#     "3:0": "3:0",
#     "3:1": "3:1",
#     "3:2": "3:2",
# }
# te same druzyny moga miec rozne nazwy w roznych sezonach z powodu np. zmiany sponsorow
TEAM_MAP = {
    "KS PAŁAC Bydgoszcz": "Pałac Bydgoszcz",
    "Bank Pocztowy Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "Polskie Przetwory Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "OnlyBio Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "Metalkas Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "Metalkas PAŁAC Bydgoszcz": "Pałac Bydgoszcz",
    "Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "KS Pałac Bydgoszcz": "Pałac Bydgoszcz",
    "Polskie Przetwory Palac Byd…": "Pałac Bydgoszcz",
    "Polskie Przetwory Pałac Byd…": "Pałac Bydgoszcz",
    "Chemik Police": "Chemik Police",
    "Grupa Azoty Chemik Police": "Chemik Police",
    "LOTTO Chemik Police": "Chemik Police",
    "Developres SkyRes Rzeszów": "Developres Rzeszów",
    "KS Developres Rzeszów": "Developres Rzeszów",
    "KS DevelopRes Rzeszów": "Developres Rzeszów",
    "Developres BELLA DOLINA Rzeszów": "Developres Rzeszów",
    "Developres BELLA DOLINA R…": "Developres Rzeszów",
    "PGE RYSICE Rzeszów": "Developres Rzeszów",
    "PGE Rysice Rzeszów": "Developres Rzeszów",
    "Grot Budowlani Łódź": "Budowlani Łódź",
    "PGE Grot Budowlani Łódź": "Budowlani Łódź",
    "Grot Budowlani Lódz": "Budowlani Łódź",
    "Budowlani Łódź": "Budowlani Łódź",
    "ŁKS Commercecon Łódź": "ŁKS Commercecon Łódź",
    "LKS Commercecon Lódz": "ŁKS Commercecon Łódź",
    "ŁKS Commerceon Łódź": "ŁKS Commercecon Łódź",
    "#VolleyWrocław": "Volley Wrocław",
    "#VolleyWroclaw": "Volley Wrocław",
    "KGHM #VolleyWrocław": "Volley Wrocław",
    "#Volley Wrocław": "Volley Wrocław",
    "Impel Wrocław": "Volley Wrocław",
    "IŁ Capital Legionovia Legion…": "Legionovia",
    "DPD IŁCapital Legionovia Le…": "Legionovia",
    "Legionovia Legionowo": "Legionovia",
    "DPD Legionovia Legionowo": "Legionovia",
    "IŁ Capital Legionovia Legionowo": "Legionovia",
    "IL Capital Legionovia Legion…": "Legionovia",
    "DPD IŁCapital Legionovia Legionowo": "Legionovia",
    "DPD ILCapital Legionovia Le…": "Legionovia",
    "E.Leclerc Radomka Radom": "Radomka Radom",
    "E.LECLERC Radomka Radom": "Radomka Radom",
    "E.LECLERC MOYA Radomka …": "Radomka Radom",
    "E.LECLERC MOYA Radomka Radom": "Radomka Radom",
    "MOYA Radomka Radom": "Radomka Radom",
    "MOYA Radomka Lotnisko Radom": "Radomka Radom",
    "MOYA Radomka Lotnisko Ra…": "Radomka Radom",
    "ITA TOOLS STAL Mielec": "Stal Mielec",
    "ITA TOOLS Stal Mielec": "Stal Mielec",
    "ITA TOOLS  STAL Mielec": "Stal Mielec",
    "ITA TOOLS  STAL Mielec": "Stal Mielec",
    "Grupa Azoty Akademia Tarnów": "Akademia Tarnów",
    "Grupa Azoty Akademia Tarnó…": "Akademia Tarnów",
    "ROLESKI GRUPA AZOTY Tarnów": "Akademia Tarnów",
    "ROLESKI GRUPA AZOTY Tarn…": "Akademia Tarnów",
    "BKS PROFI CREDIT Bielsko-Biała": "BKS Bielsko-Biała",
    "BKS STAL Bielsko-Biała": "BKS Bielsko-Biała",
    "BKS STAL Bielsko-Biala": "BKS Bielsko-Biała",
    "BKS BOSTIK Bielsko-Biała": "BKS Bielsko-Biała",
    "BKS BOSTIK ZGO Bielsko-Biała": "BKS Bielsko-Biała",
    "BKS BOSTIK ZGO Bielsko-Bia…": "BKS Bielsko-Biała",
    "BKS ALUPROF PROFI CREDIT Bielsko-Biała": "BKS Bielsko-Biała",
    "KSZO OSTROWIEC": "KSZO Ostrowiec",
    "KSZO Ostrowiec": "KSZO Ostrowiec",
    "KSZO Ostrowiec Świętokrzyski": "KSZO Ostrowiec",
    "Enea PTPS Piła": "PTPS Piła",
    "PTPS Piła": "PTPS Piła",
    "Energa MKS Kalisz": "MKS Kalisz",
    "MKS Kalisz": "MKS Kalisz",
    "UNI Opole": "UNI Opole",
    "Joker Świecie": "Joker Świecie",
    "Joker Swiecie": "Joker Świecie",
    "Sokół & Hagric Mogilno": "Sokół Mogilno",
    "Wisła Warszawa": "Wisła Warszawa",
    "Enea PTPS Pila": "PTPS Piła",
    "Atom Trefl Sopot": "Atom Trefl Sopot",
    "PGE Atom Trefl Sopot": "Atom Trefl Sopot",
    "Polski Cukier Muszynianka": "Muszynianka Muszyna",
    "Polski Cukier Muszynianka Enea": "Muszynianka Muszyna",
    "Polski Cukier Muszynianka Muszyna": "Muszynianka Muszyna",
    "Giacomini Budowlani Toruń": "Budowlani Toruń",
    "POLI Budowlani Toruń": "Budowlani Toruń",
    "Trefl Proxima Kraków": "Proxima Kraków",
    "Tauron MKS Dąbrowa Górnicza": "MKS Dąbrowa Górnicza",
    "MKS Dąbrowa Górnicza": "MKS Dąbrowa Górnicza",
}
#zmiana nazewnictwa na angielskie
RENAME_MAP = {
    "sezon": "season", "sety_A": "sets_A", "sety_B": "sets_B", "wygrana_A": "win_A",
    "A_pts_suma": "A_pts_total", "A_pts_bilans": "A_pts_balance", "A_srv_suma": "A_srv_total",
    "A_srv_bledy": "A_srv_errors", "A_srv_asy": "A_srv_aces", "A_rec_suma": "A_rec_total",
    "A_rec_bledy": "A_rec_errors", "A_rec_poz_pct": "A_rec_pos_pct", "A_atk_suma": "A_atk_total",
    "A_atk_bledy": "A_atk_errors", "A_atk_blok": "A_atk_blocked", "A_atk_pkt": "A_atk_points",
    "A_atk_skut_pct": "A_atk_success_pct", "A_blk_pkt": "A_blk_points", "B_pts_suma": "B_pts_total",
    "B_pts_bilans": "B_pts_balance", "B_srv_suma": "B_srv_total", "B_srv_bledy": "B_srv_errors",
    "B_srv_asy": "B_srv_aces", "B_rec_suma": "B_rec_total", "B_rec_bledy": "B_rec_errors",
    "B_rec_poz_pct": "B_rec_pos_pct", "B_atk_suma": "B_atk_total", "B_atk_bledy": "B_atk_errors",
    "B_atk_blok": "B_atk_blocked", "B_atk_pkt": "B_atk_points", "B_atk_skut_pct": "B_atk_success_pct",
    "B_blk_pkt": "B_blk_points", "diff_pts_suma": "diff_pts_total", "diff_pts_bilans": "diff_pts_balance",
    "diff_srv_suma": "diff_srv_total", "diff_srv_bledy": "diff_srv_errors", "diff_srv_asy": "diff_srv_aces",
    "diff_rec_suma": "diff_rec_total", "diff_rec_bledy": "diff_rec_errors", "diff_rec_poz_pct": "diff_rec_pos_pct",
    "diff_atk_suma": "diff_atk_total", "diff_atk_bledy": "diff_atk_errors", "diff_atk_blok": "diff_atk_blocked",
    "diff_atk_pkt": "diff_atk_points", "diff_atk_skut_pct": "diff_atk_success_pct", "diff_blk_pkt": "diff_blk_points",
}


## 2. Wczytanie surowych danych

Na tym etapie ładujemy surowy plik CSV, poprawiamy `match_type` i ujednolicamy nazwy drużyn.


In [62]:
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

df_raw = pd.read_csv(INPUT_FILE, sep=";")

df_raw["match_type"] = df_raw["match_type"].map(MATCH_TYPE_FIXES).fillna(df_raw["match_type"])
df_raw["team_A"] = df_raw["druzyna_A"].map(lambda value: TEAM_MAP.get(value, value))
df_raw["team_B"] = df_raw["druzyna_B"].map(lambda value: TEAM_MAP.get(value, value))

print(f"Liczba wierszy surowych: {len(df_raw)}")
print(f"Liczba kolumn surowych: {df_raw.shape[1]}")
df_raw.head()

Liczba wierszy surowych: 1616
Liczba kolumn surowych: 59


,game_id,sezon,source,match_type,druzyna_A,druzyna_B,sety_A,sety_B,wygrana_A,A_pts_suma,A_pts_bp,A_pts_bilans,A_srv_suma,A_srv_bledy,A_srv_asy,A_rec_suma,A_rec_bledy,A_rec_poz_pct,A_rec_perf_pct,A_atk_suma,A_atk_bledy,A_atk_blok,A_atk_pkt,A_atk_skut_pct,A_blk_pkt,B_pts_suma,B_pts_bp,B_pts_bilans,B_srv_suma,B_srv_bledy,B_srv_asy,B_rec_suma,B_rec_bledy,B_rec_poz_pct,B_rec_perf_pct,B_atk_suma,B_atk_bledy,B_atk_blok,B_atk_pkt,B_atk_skut_pct,B_blk_pkt,diff_pts_suma,diff_pts_bp,diff_pts_bilans,diff_srv_suma,diff_srv_bledy,diff_srv_asy,diff_rec_suma,diff_rec_bledy,diff_rec_poz_pct,diff_rec_perf_pct,diff_atk_suma,diff_atk_bledy,diff_atk_blok,diff_atk_pkt,diff_atk_skut_pct,diff_blk_pkt,team_A,team_B
0,5847,2016/2017,HTML,3:2,Polski Cukier Muszynianka,Legionovia Legionowo,3,2,1,77,41,77,103,7,10,80,11,47,26,144,14,14,56,39,11,72,30,72,96,16,11,96,10,44,20,135,10,11,47,35,14,5,11,5,7,-9,-1,-16,1,3,6,9,4,3,9,4,-3,Muszynianka Muszyna,Legionovia
1,5848,2016/2017,HTML,3:1,Chemik Police,KS PAŁAC Bydgoszcz,3,1,1,77,37,77,96,6,7,70,2,41,17,121,7,11,60,50,10,65,24,65,79,9,2,90,7,42,18,122,7,10,52,43,11,12,13,12,17,-3,5,-20,-5,-1,-1,-1,0,1,8,7,-1,Chemik Police,Pałac Bydgoszcz
2,5849,2016/2017,HTML,3:0,Legionovia Legionowo,Polski Cukier Muszynianka,0,3,0,36,12,36,52,9,2,65,11,44,15,77,10,7,29,38,5,54,28,54,73,8,11,43,2,53,18,75,3,5,36,48,7,-18,-16,-18,-21,1,-9,22,9,-9,-3,2,7,2,-7,-10,-2,Legionovia,Muszynianka Muszyna
3,5850,2016/2017,HTML,3:1,Grot Budowlani Łódź,Developres SkyRes Rzeszów,1,3,0,58,15,58,80,11,4,85,6,47,28,125,8,8,46,37,8,71,32,71,93,8,6,69,4,51,22,148,12,8,57,39,8,-13,-17,-13,-13,3,-2,16,2,-4,6,-23,-4,0,-11,-2,0,Budowlani Łódź,Developres Rzeszów
4,5851,2016/2017,HTML,3:1,Tauron MKS Dąbrowa Górnicza,KSZO OSTROWIEC,3,1,1,75,37,75,98,10,5,84,4,49,34,150,15,11,55,37,15,63,27,63,94,10,4,88,5,54,43,148,10,15,48,32,11,12,10,12,4,0,1,-4,-1,-5,-9,2,5,-4,7,5,4,MKS Dąbrowa Górnicza,KSZO Ostrowiec


## 3. Kontrola nazw drużyn i podstawowa walidacja

Tutaj sprawdzamy, czy zostały jeszcze jakieś nazwy drużyn do dopisania do mapy oraz czy `game_id` wygląda sensownie.


In [63]:
unknown_teams = sorted(set(df_raw["druzyna_A"]).union(df_raw["druzyna_B"]) - set(TEAM_MAP))
print("Nieznane nazwy drużyn:")
print(unknown_teams if unknown_teams else "Brak")

print(f"Duplikaty game_id po wczytaniu : {df_raw['game_id'].duplicated().sum()}")
df_raw[["game_id", "team_A", "team_B", "match_type"]].head(10)


Nieznane nazwy drużyn:
Brak
Duplikaty game_id po wczytaniu : 0


,game_id,team_A,team_B,match_type
0,5847,Muszynianka Muszyna,Legionovia,3:2
1,5848,Chemik Police,Pałac Bydgoszcz,3:1
2,5849,Legionovia,Muszynianka Muszyna,3:0
3,5850,Budowlani Łódź,Developres Rzeszów,3:1
4,5851,MKS Dąbrowa Górnicza,KSZO Ostrowiec,3:1
5,5852,Volley Wrocław,ŁKS Commercecon Łódź,3:0
6,5853,Budowlani Toruń,BKS Bielsko-Biała,3:2
7,5854,Atom Trefl Sopot,Budowlani Toruń,3:0
8,5855,BKS Bielsko-Biała,Volley Wrocław,3:1
9,5856,ŁKS Commercecon Łódź,MKS Dąbrowa Górnicza,3:1


## 4. Feature engineering na surowych kolumnach

Dodajemy wskaźniki efektywności, rolling features i zmienn. Liczymy dla drużyny A, drużyny B oraz dla różnicy między drużynami


In [64]:
df_features = df_raw.copy()
# efektywność serwisu: stosunek asów do błędów
df_features["A_srv_eff"] = (df_features["A_srv_asy"] / df_features["A_srv_bledy"].replace(0, np.nan)).fillna(df_features["A_srv_asy"])
df_features["B_srv_eff"] = (df_features["B_srv_asy"] / df_features["B_srv_bledy"].replace(0, np.nan)).fillna(df_features["B_srv_asy"])
df_features["diff_srv_eff"] = df_features["A_srv_eff"] - df_features["B_srv_eff"]

# efektywność ataku
df_features["A_atk_eff"] = ((df_features["A_atk_pkt"] - df_features["A_atk_bledy"] - df_features["A_atk_blok"]) / df_features["A_atk_suma"].replace(0, np.nan)).fillna(0)
df_features["B_atk_eff"] = ((df_features["B_atk_pkt"] - df_features["B_atk_bledy"] - df_features["B_atk_blok"]) / df_features["B_atk_suma"].replace(0, np.nan)).fillna(0)
df_features["diff_atk_eff"] = df_features["A_atk_eff"] - df_features["B_atk_eff"]

# udział bloku i serwisu w dorobku punktowym
df_features["A_blk_share"] = (df_features["A_blk_pkt"] / df_features["A_pts_suma"].replace(0, np.nan)).fillna(0)
df_features["B_blk_share"] = (df_features["B_blk_pkt"] / df_features["B_pts_suma"].replace(0, np.nan)).fillna(0)
df_features["diff_blk_share"] = df_features["A_blk_share"] - df_features["B_blk_share"]
df_features["A_srv_share"] = (df_features["A_srv_asy"] / df_features["A_pts_suma"].replace(0, np.nan)).fillna(0)
df_features["B_srv_share"] = (df_features["B_srv_asy"] / df_features["B_pts_suma"].replace(0, np.nan)).fillna(0)
df_features["diff_srv_share"] = df_features["A_srv_share"] - df_features["B_srv_share"]

# suma błędów i punkty z tablicy
df_features["A_errors_total"] = df_features["A_srv_bledy"] + df_features["A_atk_bledy"] + df_features["A_rec_bledy"]
df_features["B_errors_total"] = df_features["B_srv_bledy"] + df_features["B_atk_bledy"] + df_features["B_rec_bledy"]
df_features["diff_errors_total"] = df_features["A_errors_total"] - df_features["B_errors_total"]
df_features["A_scoreboard_points"] = df_features["A_pts_suma"] + df_features["B_errors_total"]
df_features["B_scoreboard_points"] = df_features["B_pts_suma"] + df_features["A_errors_total"]
df_features["match_total_points"] = df_features["A_scoreboard_points"] + df_features["B_scoreboard_points"]

# błędy na akcję
a_actions = df_features["A_srv_suma"] + df_features["A_atk_suma"] + df_features["A_rec_suma"]
b_actions = df_features["B_srv_suma"] + df_features["B_atk_suma"] + df_features["B_rec_suma"]
df_features["A_errors_rate"] = (df_features["A_errors_total"] / a_actions.replace(0, np.nan)).fillna(0)
df_features["B_errors_rate"] = (df_features["B_errors_total"] / b_actions.replace(0, np.nan)).fillna(0)
df_features["diff_errors_rate"] = df_features["A_errors_rate"] - df_features["B_errors_rate"]

# różnice jakości przyjęcia
df_features["A_rec_quality_gap"] = df_features["A_rec_perf_pct"] - df_features["A_rec_poz_pct"]
df_features["B_rec_quality_gap"] = df_features["B_rec_perf_pct"] - df_features["B_rec_poz_pct"]
df_features["diff_rec_quality_gap"] = df_features["A_rec_quality_gap"] - df_features["B_rec_quality_gap"]

# zmienne na rozegrany set
df_features["sets_sum"] = df_features["sety_A"] + df_features["sety_B"]
df_features["loser_sets"] = df_features[["sety_A", "sety_B"]].min(axis=1)
df_features["A_pts_per_set"] = (df_features["A_pts_suma"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["B_pts_per_set"] = (df_features["B_pts_suma"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["diff_pts_per_set"] = df_features["A_pts_per_set"] - df_features["B_pts_per_set"]
df_features["A_scoreboard_pts_per_set"] = (df_features["A_scoreboard_points"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["B_scoreboard_pts_per_set"] = (df_features["B_scoreboard_points"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["diff_scoreboard_pts_per_set"] = df_features["A_scoreboard_pts_per_set"] - df_features["B_scoreboard_pts_per_set"]
df_features["match_avg_points_per_set"] = (df_features["match_total_points"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["A_errors_per_set"] = (df_features["A_errors_total"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["B_errors_per_set"] = (df_features["B_errors_total"] / df_features["sets_sum"].replace(0, np.nan)).fillna(0)
df_features["diff_errors_per_set"] = df_features["A_errors_per_set"] - df_features["B_errors_per_set"]

# skuteczność zagrywki i obrona
df_features["A_srv_success_pct"] = (df_features["A_srv_asy"] / df_features["A_srv_suma"].replace(0, np.nan) * 100).fillna(0)
df_features["B_srv_success_pct"] = (df_features["B_srv_asy"] / df_features["B_srv_suma"].replace(0, np.nan) * 100).fillna(0)
df_features["diff_srv_success_pct"] = df_features["A_srv_success_pct"] - df_features["B_srv_success_pct"]
df_features["A_defense_total"] = df_features["A_rec_suma"] + df_features["A_atk_blok"]
df_features["B_defense_total"] = df_features["B_rec_suma"] + df_features["B_atk_blok"]
df_features["diff_defense_total"] = df_features["A_defense_total"] - df_features["B_defense_total"]

# rolling średnie - surowe i na set
df_features = df_features.sort_values(["team_A", "game_id"])
df_features["A_pts_rolling3"] = df_features.groupby("team_A")["A_pts_suma"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["A_errors_rolling3"] = df_features.groupby("team_A")["A_errors_total"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["A_pts_per_set_rolling3"] = df_features.groupby("team_A")["A_pts_per_set"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["A_errors_per_set_rolling3"] = df_features.groupby("team_A")["A_errors_per_set"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

df_features = df_features.sort_values(["team_B", "game_id"])
df_features["B_pts_rolling3"] = df_features.groupby("team_B")["B_pts_suma"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["B_errors_rolling3"] = df_features.groupby("team_B")["B_errors_total"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["B_pts_per_set_rolling3"] = df_features.groupby("team_B")["B_pts_per_set"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["B_errors_per_set_rolling3"] = df_features.groupby("team_B")["B_errors_per_set"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_features["diff_pts_rolling3"] = df_features["A_pts_rolling3"] - df_features["B_pts_rolling3"]
df_features["diff_errors_rolling3"] = df_features["A_errors_rolling3"] - df_features["B_errors_rolling3"]
df_features["diff_pts_per_set_rolling3"] = df_features["A_pts_per_set_rolling3"] - df_features["B_pts_per_set_rolling3"]
df_features["diff_errors_per_set_rolling3"] = df_features["A_errors_per_set_rolling3"] - df_features["B_errors_per_set_rolling3"]

print(f"Liczba kolumn po feature engineering: {df_features.shape[1]}")
df_features[["game_id", "team_A", "team_B", "sets_sum", "loser_sets", "A_pts_per_set", "B_pts_per_set", "A_scoreboard_points", "B_scoreboard_points", "match_total_points", "match_avg_points_per_set", "A_pts_per_set_rolling3", "B_pts_per_set_rolling3", "diff_pts_per_set_rolling3", "A_errors_per_set_rolling3", "B_errors_per_set_rolling3", "diff_errors_per_set_rolling3"]].head()

Liczba kolumn po feature engineering: 113


,game_id,team_A,team_B,sets_sum,loser_sets,A_pts_per_set,B_pts_per_set,A_scoreboard_points,B_scoreboard_points,match_total_points,match_avg_points_per_set,A_pts_per_set_rolling3,B_pts_per_set_rolling3,diff_pts_per_set_rolling3,A_errors_per_set_rolling3,B_errors_per_set_rolling3,diff_errors_per_set_rolling3
1008,1102043,Pałac Bydgoszcz,Akademia Tarnów,4,1,18.250000,12.25,102,75,177,44.250000,15.250000,NaN,NaN,5.550000,NaN,NaN
1016,1102052,Developres Rzeszów,Akademia Tarnów,3,0,20.666667,13.00,75,56,131,43.666667,16.050000,12.250000,3.800000,4.783333,7.250000,-2.466667
1023,1102061,Radomka Radom,Akademia Tarnów,5,2,17.800000,19.00,116,111,227,45.400000,17.055556,12.625000,4.430556,5.572222,5.791667,-0.219444
1038,1102080,Budowlani Łódź,Akademia Tarnów,5,2,16.600000,14.00,111,96,207,41.400000,16.061111,14.750000,1.311111,5.227778,5.661111,-0.433333
1061,1102110,MKS Kalisz,Akademia Tarnów,5,2,15.000000,14.20,105,101,206,41.200000,15.816667,15.333333,0.483333,6.500000,5.111111,1.388889


## 5. Finalizacja datasetu

Tutaj zmieniamy nazwy kolumn na finalne, usuwamy zbędne pola i ustawiamy kolejność kolumn.


In [65]:
df_final = df_features.drop(columns=["druzyna_A", "druzyna_B", "source"], errors="ignore").rename(columns=RENAME_MAP).copy()
leading_columns = ["game_id", "season", "team_A", "team_B", "match_type", "sets_A", "sets_B", "win_A"]
remaining_columns = [column for column in df_final.columns if column not in leading_columns]
df_final = df_final[leading_columns + remaining_columns].sort_values(["season", "game_id"]).reset_index(drop=True)

print(f"Liczba kolumn po finalizacji: {df_final.shape[1]}")
print(df_final.columns.tolist())
df_final.head()

Liczba kolumn po finalizacji: 110
['game_id', 'season', 'team_A', 'team_B', 'match_type', 'sets_A', 'sets_B', 'win_A', 'A_pts_total', 'A_pts_bp', 'A_pts_balance', 'A_srv_total', 'A_srv_errors', 'A_srv_aces', 'A_rec_total', 'A_rec_errors', 'A_rec_pos_pct', 'A_rec_perf_pct', 'A_atk_total', 'A_atk_errors', 'A_atk_blocked', 'A_atk_points', 'A_atk_success_pct', 'A_blk_points', 'B_pts_total', 'B_pts_bp', 'B_pts_balance', 'B_srv_total', 'B_srv_errors', 'B_srv_aces', 'B_rec_total', 'B_rec_errors', 'B_rec_pos_pct', 'B_rec_perf_pct', 'B_atk_total', 'B_atk_errors', 'B_atk_blocked', 'B_atk_points', 'B_atk_success_pct', 'B_blk_points', 'diff_pts_total', 'diff_pts_bp', 'diff_pts_balance', 'diff_srv_total', 'diff_srv_errors', 'diff_srv_aces', 'diff_rec_total', 'diff_rec_errors', 'diff_rec_pos_pct', 'diff_rec_perf_pct', 'diff_atk_total', 'diff_atk_errors', 'diff_atk_blocked', 'diff_atk_points', 'diff_atk_success_pct', 'diff_blk_points', 'A_srv_eff', 'B_srv_eff', 'diff_srv_eff', 'A_atk_eff', 'B_atk_eff

,game_id,season,team_A,team_B,match_type,sets_A,sets_B,win_A,A_pts_total,A_pts_bp,A_pts_balance,A_srv_total,A_srv_errors,A_srv_aces,A_rec_total,A_rec_errors,A_rec_pos_pct,A_rec_perf_pct,A_atk_total,A_atk_errors,A_atk_blocked,A_atk_points,A_atk_success_pct,A_blk_points,B_pts_total,B_pts_bp,B_pts_balance,B_srv_total,B_srv_errors,B_srv_aces,B_rec_total,B_rec_errors,B_rec_pos_pct,B_rec_perf_pct,B_atk_total,B_atk_errors,B_atk_blocked,B_atk_points,B_atk_success_pct,B_blk_points,diff_pts_total,diff_pts_bp,diff_pts_balance,diff_srv_total,diff_srv_errors,diff_srv_aces,diff_rec_total,diff_rec_errors,diff_rec_pos_pct,diff_rec_perf_pct,diff_atk_total,diff_atk_errors,diff_atk_blocked,diff_atk_points,diff_atk_success_pct,diff_blk_points,A_srv_eff,B_srv_eff,diff_srv_eff,A_atk_eff,B_atk_eff,diff_atk_eff,A_blk_share,B_blk_share,diff_blk_share,A_srv_share,B_srv_share,diff_srv_share,A_errors_total,B_errors_total,diff_errors_total,A_scoreboard_points,B_scoreboard_points,match_total_points,A_errors_rate,B_errors_rate,diff_errors_rate,A_rec_quality_gap,B_rec_quality_gap,diff_rec_quality_gap,sets_sum,loser_sets,A_pts_per_set,B_pts_per_set,diff_pts_per_set,A_scoreboard_pts_per_set,B_scoreboard_pts_per_set,diff_scoreboard_pts_per_set,match_avg_points_per_set,A_errors_per_set,B_errors_per_set,diff_errors_per_set,A_srv_success_pct,B_srv_success_pct,diff_srv_success_pct,A_defense_total,B_defense_total,diff_defense_total,A_pts_rolling3,A_errors_rolling3,A_pts_per_set_rolling3,A_errors_per_set_rolling3,B_pts_rolling3,B_errors_rolling3,B_pts_per_set_rolling3,B_errors_per_set_rolling3,diff_pts_rolling3,diff_errors_rolling3,diff_pts_per_set_rolling3,diff_errors_per_set_rolling3
0,5365,2015/2016,Chemik Police,Pałac Bydgoszcz,3:0,3,0,1,60,32,60,73,8,4,41,4,49,12,85,3,5,40,47,16,36,9,36,50,9,4,65,4,49,8,85,2,16,27,32,5,24,23,24,23,-1,0,-24,0,0,4,0,1,-11,13,15,11,0.500,0.444444,0.055556,0.376471,0.105882,0.270588,0.266667,0.138889,0.127778,0.066667,0.111111,-0.044444,15,15,0,75,51,126,0.075377,0.075000,0.000377,-37,-41,4,3,0,20.000000,12.000000,8.000000,25.0,17.000000,8.000000,42.000000,5.000000,5.000000,0.000000,5.479452,8.000000,-2.520548,46,81,-35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5366,2015/2016,PTPS Piła,Volley Wrocław,3:1,1,3,0,54,23,54,85,6,3,88,16,53,21,147,8,8,43,29,8,79,37,79,97,9,8,79,6,49,11,169,13,8,63,37,8,-25,-14,-25,-12,-3,-5,9,10,4,10,-22,-5,0,-20,-8,2,0.500,0.888889,-0.388889,0.183673,0.248521,-0.064847,0.148148,0.101266,0.046882,0.055556,0.101266,-0.045710,30,28,2,82,109,191,0.093750,0.081159,0.012591,-32,-38,6,4,1,13.500000,19.750000,-6.250000,20.5,27.250000,-6.750000,47.750000,7.500000,7.000000,0.500000,3.529412,8.247423,-4.718011,96,87,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5367,2015/2016,Legionovia,Chemik Police,3:0,0,3,0,29,10,29,50,5,2,64,3,29,11,85,8,10,21,25,6,56,30,56,74,10,3,45,2,38,29,97,7,6,43,44,10,-27,-20,-27,-24,-5,-1,19,1,-9,-18,-12,1,4,-22,-19,-4,0.400,0.300000,0.100000,0.035294,0.309278,-0.273984,0.206897,0.178571,0.028325,0.068966,0.053571,0.015394,16,19,-3,48,72,120,0.080402,0.087963,-0.007561,-18,-9,-9,3,0,9.666667,18.666667,-9.000000,16.0,24.000000,-8.000000,40.000000,5.333333,6.333333,-1.000000,4.000000,4.054054,-0.054054,74,51,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5368,2015/2016,MKS Dąbrowa Górnicza,Atom Trefl Sopot,3:2,2,3,0,76,31,76,102,16,6,98,12,33,14,161,8,7,62,39,8,77,37,77,110,12,12,86,6,41,23,159,11,8,58,36,7,-1,-6,-1,-8,4,-6,12,6,-8,-9,2,-3,-1,4,3,1,0.375,1.000000,-0.625000,0.291925,0.245283,0.046642,0.105263,0.090909,0.014354,0.078947,0.155844,-0.076897,36,29,7,105,113,218,0.099723,0.081690,0.018033,-19,-18,-1,5,2,15.200000,15.400000,-0.200000,21.0,22.600000,-1.600000,43.600000,7.200000,5.800000,1.400000,5.882353,10.909091,-5.026738,105,94,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5369,2015/2016,Volley Wrocław,Muszynianka Muszyna,3:0,3,0,1,56,30,56,74,8,5,53,3,51,22,89,6,5,39,44,12,39,16,39,60,7,3,66,5,45,27,103,10,12,31,30,5,17,14,17,14,1,2,-13,-2,6,-5,-14,

## 6. Dodanie `czy_playoff` i cech historycznych

Na tym etapie budujemy flagę playoff oraz wszystkie cechy prematch/history.


In [66]:
df_model = df_final.copy().sort_values(["season", "game_id"]).reset_index(drop=True)

pair_counts = {}
playoff_flags = []
for _, row in df_model.iterrows():
    pair = tuple(sorted((str(row["team_A"]), str(row["team_B"]))))
    key = (str(row["season"]), pair)
    pair_counts[key] = pair_counts.get(key, 0) + 1
    playoff_flags.append(1 if pair_counts[key] >= 3 else 0)

df_model["czy_playoff"] = playoff_flags

team_games_a = pd.DataFrame({
    "season": df_model["season"], "game_id": df_model["game_id"], "team": df_model["team_A"],
    "team_role": "A", "win": df_model["win_A"], "pts_total": df_model["A_pts_total"],
    "errors_total": df_model["A_errors_total"], "atk_eff": df_model["A_atk_eff"],
})
team_games_b = pd.DataFrame({
    "season": df_model["season"], "game_id": df_model["game_id"], "team": df_model["team_B"],
    "team_role": "B", "win": 1 - df_model["win_A"], "pts_total": df_model["B_pts_total"],
    "errors_total": df_model["B_errors_total"], "atk_eff": df_model["B_atk_eff"],
})
team_games = pd.concat([team_games_a, team_games_b], ignore_index=True)
team_games = team_games.sort_values(["season", "team", "game_id", "team_role"]).reset_index(drop=True)

grouped = team_games.groupby(["season", "team"], sort=False)
team_games["matches_before"] = grouped.cumcount()
team_games["win_rate_prev"] = grouped["win"].transform(lambda s: s.shift(1).expanding().mean())
team_games["win_rate_last5"] = grouped["win"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
team_games["pts_prev_all"] = grouped["pts_total"].transform(lambda s: s.shift(1).expanding().mean())
team_games["errors_prev_all"] = grouped["errors_total"].transform(lambda s: s.shift(1).expanding().mean())
team_games["atk_eff_prev_all"] = grouped["atk_eff"].transform(lambda s: s.shift(1).expanding().mean())
team_games["win_streak_prev3"] = grouped["win"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

team_features = team_games[["season", "game_id", "team", "matches_before", "win_rate_prev", "win_rate_last5", "pts_prev_all", "errors_prev_all", "atk_eff_prev_all", "win_streak_prev3"]]
a_features = team_features.rename(columns={"team": "team_A", "matches_before": "A_matches_before", "win_rate_prev": "A_win_rate_prev", "win_rate_last5": "A_win_rate_last5", "pts_prev_all": "A_pts_prev_all", "errors_prev_all": "A_errors_prev_all", "atk_eff_prev_all": "A_atk_eff_prev_all", "win_streak_prev3": "A_win_streak_prev3"})
b_features = team_features.rename(columns={"team": "team_B", "matches_before": "B_matches_before", "win_rate_prev": "B_win_rate_prev", "win_rate_last5": "B_win_rate_last5", "pts_prev_all": "B_pts_prev_all", "errors_prev_all": "B_errors_prev_all", "atk_eff_prev_all": "B_atk_eff_prev_all", "win_streak_prev3": "B_win_streak_prev3"})

df_model = df_model.merge(a_features, on=["season", "game_id", "team_A"], how="left")
df_model = df_model.merge(b_features, on=["season", "game_id", "team_B"], how="left")
for suffix in ["matches_before", "win_rate_prev", "win_rate_last5", "pts_prev_all", "errors_prev_all", "atk_eff_prev_all", "win_streak_prev3"]:
    df_model[f"diff_{suffix}"] = df_model[f"A_{suffix}"] - df_model[f"B_{suffix}"]

h2h_state_season = {}
h2h_state_all = {}
h2h_matches_before, h2h_win_rate_A, h2h_last_result_A = [], [], []
h2h_matches_before_all, h2h_win_rate_A_all, h2h_last_result_A_all = [], [], []

for _, row in df_model.iterrows():
    team_a = str(row["team_A"])
    team_b = str(row["team_B"])
    pair = tuple(sorted((team_a, team_b)))
    key_season = (str(row["season"]), pair)
    state_season = h2h_state_season.setdefault(key_season, {"matches": 0, "wins": {}, "last_winner": None})
    state_all = h2h_state_all.setdefault(pair, {"matches": 0, "wins": {}, "last_winner": None})

    for state, matches_list, win_rate_list, last_result_list in [
        (state_season, h2h_matches_before, h2h_win_rate_A, h2h_last_result_A),
        (state_all, h2h_matches_before_all, h2h_win_rate_A_all, h2h_last_result_A_all),
    ]:
        matches_before = int(state["matches"])
        wins_a = int(state["wins"].get(team_a, 0))
        matches_list.append(matches_before)
        win_rate_list.append(wins_a / matches_before if matches_before > 0 else 0.5)
        last_winner = state["last_winner"]
        last_result_list.append(0.5 if last_winner is None else (1.0 if last_winner == team_a else 0.0))

    winner = team_a if int(row["win_A"]) == 1 else team_b
    for state in [state_season, state_all]:
        state["wins"][winner] = int(state["wins"].get(winner, 0)) + 1
        state["matches"] = int(state["matches"]) + 1
        state["last_winner"] = winner

df_model["h2h_matches_before"] = h2h_matches_before
df_model["h2h_win_rate_A"] = h2h_win_rate_A
df_model["h2h_last_result_A"] = h2h_last_result_A
df_model["h2h_matches_before_all"] = h2h_matches_before_all
df_model["h2h_win_rate_A_all"] = h2h_win_rate_A_all
df_model["h2h_last_result_A_all"] = h2h_last_result_A_all

print(f"Liczba wierszy: {len(df_model)}")
print(f"Liczba kolumn po historii: {df_model.shape[1]}")
df_model[["game_id", "season", "team_A", "team_B", "sets_sum", "loser_sets", "match_total_points", "match_avg_points_per_set", "czy_playoff", "A_pts_per_set_rolling3", "B_pts_per_set_rolling3", "diff_pts_per_set_rolling3", "A_errors_per_set_rolling3", "B_errors_per_set_rolling3", "diff_errors_per_set_rolling3", "A_matches_before", "B_matches_before", "diff_matches_before", "h2h_matches_before", "h2h_matches_before_all"]].head(10)

Liczba wierszy: 1616
Liczba kolumn po historii: 138


,game_id,season,team_A,team_B,sets_sum,loser_sets,match_total_points,match_avg_points_per_set,czy_playoff,A_pts_per_set_rolling3,B_pts_per_set_rolling3,diff_pts_per_set_rolling3,A_errors_per_set_rolling3,B_errors_per_set_rolling3,diff_errors_per_set_rolling3,A_matches_before,B_matches_before,diff_matches_before,h2h_matches_before,h2h_matches_before_all
0,5365,2015/2016,Chemik Police,Pałac Bydgoszcz,3,0,126,42.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,5366,2015/2016,PTPS Piła,Volley Wrocław,4,1,191,47.750000,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2,5367,2015/2016,Legionovia,Chemik Police,3,0,120,40.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,0,1,-1,0,0
3,5368,2015/2016,MKS Dąbrowa Górnicza,Atom Trefl Sopot,5,2,218,43.600000,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
4,5369,2015/2016,Volley Wrocław,Muszynianka Muszyna,3,0,134,44.666667,0,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,0,0
5,5370,2015/2016,Pałac Bydgoszcz,Volley Wrocław,3,0,136,45.333333,0,NaN,19.75,NaN,NaN,7.0,NaN,1,2,-1,0,0
6,5371,2015/2016,Muszynianka Muszyna,MKS Dąbrowa Górnicza,3,0,149,49.666667,0,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0,0
7,5372,2015/2016,Atom Trefl Sopot,Legionovia,3,0,141,47.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0,0
8,5373,2015/2016,Chemik Police,Budowlani Łódź,3,0,131,43.666667,0,20.0,NaN,NaN,5.0,NaN,NaN,2,0,2,0,0
9,5374,2015/2016,Developres Rzeszów,BKS Bielsko-Biała,3,0,122,40.666667,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0


## 7. Walidacja końcowa

Sprawdzamy duplikaty, braki w najważniejszych kolumnach oraz szybki podgląd rozkładu `czy_playoff`.


In [67]:
key_columns = [
    "game_id", "season", "team_A", "team_B", "match_type", "win_A", "czy_playoff",
    "sets_sum", "loser_sets", "match_avg_points_per_set",
    "diff_atk_eff", "diff_srv_eff", "diff_errors_total", "diff_pts_per_set_rolling3", "diff_errors_per_set_rolling3",
]

print("Duplikaty game_id:", df_model["game_id"].duplicated().sum())
print("Braki w kluczowych kolumnach:")
print(df_model[key_columns].isnull().sum())
print("\nRozkład czy_playoff:")
print(df_model["czy_playoff"].value_counts(dropna=False).to_string())

df_model[key_columns].head()

Duplikaty game_id: 0
Braki w kluczowych kolumnach:
game_id                          0
season                           0
team_A                           0
team_B                           0
match_type                       0
win_A                            0
czy_playoff                      0
sets_sum                         0
loser_sets                       0
match_avg_points_per_set         0
diff_atk_eff                     0
diff_srv_eff                     0
diff_errors_total                0
diff_pts_per_set_rolling3       37
diff_errors_per_set_rolling3    37
dtype: int64

Rozkład czy_playoff:
czy_playoff
0    1393
1     223


,game_id,season,team_A,team_B,match_type,win_A,czy_playoff,sets_sum,loser_sets,match_avg_points_per_set,diff_atk_eff,diff_srv_eff,diff_errors_total,diff_pts_per_set_rolling3,diff_errors_per_set_rolling3
0,5365,2015/2016,Chemik Police,Pałac Bydgoszcz,3:0,1,0,3,0,42.000000,0.270588,0.055556,0,NaN,NaN
1,5366,2015/2016,PTPS Piła,Volley Wrocław,3:1,0,0,4,1,47.750000,-0.064847,-0.388889,2,NaN,NaN
2,5367,2015/2016,Legionovia,Chemik Police,3:0,0,0,3,0,40.000000,-0.273984,0.100000,-3,NaN,NaN
3,5368,2015/2016,MKS Dąbrowa Górnicza,Atom Trefl Sopot,3:2,0,0,5,2,43.600000,0.046642,-0.625000,7,NaN,NaN
4,5369,2015/2016,Volley Wrocław,Muszynianka Muszyna,3:0,1,0,3,0,44.666667,0.227228,0.196429,-5,NaN,NaN


## 8. Zapis gotowego pliku



In [68]:
df_model.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Zapisano {len(df_model)} wierszy i {len(df_model.columns)} kolumn do {OUTPUT_FILE}.")
OUTPUT_FILE


Zapisano 1616 wierszy i 138 kolumn do processed_data.csv.


PosixPath('processed_data.csv')